# 실습 9: Ollama LLM을 활용한 HTTP 분류 (2교시)

특성 추출 없이 **HTTP 요청 텍스트를 그대로** Ollama gemma3:4b에 보여주고
정상/공격을 분류합니다.

**사전 조건**:
- Ollama 서버 실행 중 (`ollama serve`)
- `gemma3:4b` 모델 다운로드됨 (`ollama list`로 확인)
- 7주차 `processed_data.pkl` 존재 (LLM용 텍스트 샘플 포함)

In [8]:
import sys
!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install ollama

In [9]:
# %% [Setup] 패키지 import 및 LLM용 샘플 로드
import pickle
import time
import json
import re
import pandas as pd
from urllib.parse import unquote
from sklearn.metrics import accuracy_score, f1_score, classification_report

import ollama  # pip install ollama

with open("../ch7/processed_data.pkl", "rb") as f:  # 경로 수정
    data = pickle.load(f)

llm_sample = data["llm_sample"].head(100).reset_index(drop=True)
print(f"분류 대상: {len(llm_sample)}건")
print(f"라벨 분포: 정상 {(llm_sample.get('is_attack',0)==0).sum()}건 / "
      f"공격 {(llm_sample.get('is_attack',0)==1).sum()}건")

분류 대상: 100건
라벨 분포: 정상 47건 / 공격 53건


## 1. 분류 프롬프트 설계

LLM 응답을 안정적으로 파싱하기 위해:
- **Few-shot 예시** 2개로 출력 형식을 학습시킴
- **JSON 형태**로 응답하도록 강제
- 영어 프롬프트(gemma3:4b가 영어에 더 정확)

In [10]:
# %% [1] HTTP 텍스트 재구성 + 공통 유틸
def build_http_text(row) -> str:
    method = row.get("method", "GET")
    url    = unquote(str(row.get("url", "")), encoding="latin-1")
    body   = str(row.get("body_decoded", row.get("body", "")) or "")
    text   = f"{method} {url} HTTP/1.1"
    if body and body != "nan":
        text += f"\nBody: {body[:200]}"
    return text


def parse_llm_response(text: str) -> dict:
    """LLM 응답에서 JSON 추출"""
    match = re.search(r"\{[^{}]*\}", text, re.DOTALL)
    if not match:
        return {"label": "Unknown", "reason": text[:80]}
    try:
        return json.loads(match.group())
    except json.JSONDecodeError:
        return {"label": "Unknown", "reason": text[:80]}


def run_classification(prompt_fn, llm_sample, model="gemma3:4b", label=""):
    """프롬프트 함수로 전체 샘플 분류 후 결과 반환"""
    results = []
    start = time.time()
    for i, row in llm_sample.iterrows():
        http_text = build_http_text(row)
        prompt = prompt_fn(http_text)
        resp = ollama.chat(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            options={"temperature": 0},
        )
        parsed = parse_llm_response(resp["message"]["content"])
        true_label = "Anomalous" if row.get("is_attack", 0) == 1 else "Normal"
        results.append({
            "idx": i,
            "true": true_label,
            "pred": parsed.get("label", "Unknown"),
            "reason": parsed.get("reason", "")[:120],
            "http_short": http_text[:100],
        })
        if (i + 1) % 10 == 0:
            elapsed = time.time() - start
            print(f"  [{label}] {i+1}/{len(llm_sample)}건 완료 "
                  f"({elapsed:.1f}초, 건당 {elapsed/(i+1):.2f}초)")

    elapsed = time.time() - start
    df = pd.DataFrame(results)
    df["pred_clean"] = df["pred"].replace({"Unknown": "Normal"})
    y_true = (df["true"] == "Anomalous").astype(int)
    y_pred = (df["pred_clean"] == "Anomalous").astype(int)
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, zero_division=0)
    unknown_cnt = (df["pred"] == "Unknown").sum()
    print(f"\n[{label}] 총 {elapsed:.1f}초 | Acc={acc:.4f} | F1={f1:.4f} | Unknown={unknown_cnt}건")
    return df, acc, f1, elapsed, unknown_cnt


print("유틸 함수 정의 완료")

유틸 함수 정의 완료


## 2. 프롬프트 전략 4종 정의

| 전략 | 설명 |
|------|------|
| **A. 2-shot Basic** | 원본 프롬프트 (기준선) |
| **B. 5-shot Multi-attack** | SQL·XSS·Path Traversal·Command Injection 예시 추가 |
| **C. Chain-of-Thought** | 단계별 추론 후 판단 |
| **D. Rule-hint** | 탐지 규칙 힌트 포함 + 5-shot |

In [11]:
# %% [2] 프롬프트 전략 4종 정의

# ── A. 2-shot Basic (기준선, 원본) ─────────────────────────────────────────
def prompt_a(http_text: str) -> str:
    return (
        'You are a web security expert. Classify each HTTP request as "Normal" or "Anomalous" '
        'and provide a brief reason.\n\n'
        'Examples:\n'
        'Request: GET /index.jsp HTTP/1.1\n'
        'Output: {{"label": "Normal", "reason": "Standard page request, no suspicious pattern"}}\n\n'
        "Request: GET /search?q=' OR '1'='1 HTTP/1.1\n"
        'Output: {{"label": "Anomalous", "reason": "Classic SQL Injection pattern with OR 1=1"}}\n\n'
        f'Now classify:\nRequest: {http_text}\nOutput:'
    )


# ── B. 5-shot Multi-attack ──────────────────────────────────────────────────
def prompt_b(http_text: str) -> str:
    return (
        'You are a web security expert. Classify the HTTP request as "Normal" or "Anomalous".\n'
        'Respond ONLY with valid JSON: {"label": "...", "reason": "..."}\n\n'
        '### Examples\n'
        'Request: GET /tienda1/publico/index.jsp HTTP/1.1\n'
        'Output: {"label": "Normal", "reason": "Regular page navigation, no attack indicators"}\n\n'
        "Request: GET /search?q=1' OR '1'='1 HTTP/1.1\n"
        'Output: {"label": "Anomalous", "reason": "SQL Injection via OR-based tautology"}\n\n'
        'Request: GET /page?name=<script>alert(1)</script> HTTP/1.1\n'
        'Output: {"label": "Anomalous", "reason": "XSS: injected <script> tag in parameter"}\n\n'
        'Request: GET /files?path=../../etc/passwd HTTP/1.1\n'
        'Output: {"label": "Anomalous", "reason": "Path Traversal: ../.. to access /etc/passwd"}\n\n'
        'Request: POST /upload HTTP/1.1\nBody: filename=shell.php&cmd=ls\n'
        'Output: {"label": "Anomalous", "reason": "Command Injection via cmd parameter"}\n\n'
        f'### Classify this\nRequest: {http_text}\nOutput:'
    )


# ── C. Chain-of-Thought ─────────────────────────────────────────────────────
def prompt_c(http_text: str) -> str:
    return (
        'You are a web security analyst. Think step by step before classifying.\n\n'
        '### Example\n'
        "Request: GET /search?q=1' OR '1'='1 HTTP/1.1\n"
        'Step 1 - Parse: URL parameter q contains single quotes and OR keyword.\n'
        'Step 2 - Pattern: OR-based tautology is classic SQL Injection.\n'
        'Step 3 - Decision: Anomalous.\n'
        'Output: {"label": "Anomalous", "reason": "SQL Injection via OR tautology in q param"}\n\n'
        'Request: GET /tienda1/publico/index.jsp HTTP/1.1\n'
        'Step 1 - Parse: standard JSP path, no parameters.\n'
        'Step 2 - Pattern: no special characters or known attack signatures.\n'
        'Step 3 - Decision: Normal.\n'
        'Output: {"label": "Normal", "reason": "Plain page request with no suspicious elements"}\n\n'
        f'Now analyze step by step and output JSON only:\nRequest: {http_text}\nOutput:'
    )


# ── D. Rule-hint + 5-shot ───────────────────────────────────────────────────
def prompt_d(http_text: str) -> str:
    return (
        'You are a web security expert. Classify HTTP requests as "Normal" or "Anomalous".\n\n'
        '### Attack indicators to look for:\n'
        "- SQL Injection: ' OR, UNION SELECT, 1=1, --, ;DROP\n"
        '- XSS: <script>, onerror=, javascript:, alert(\n'
        '- Path Traversal: ../, %2e%2e, /etc/passwd, /windows/system32\n'
        '- Command Injection: ;ls, |whoami, `id`, %0a\n'
        '- Encoding evasion: %27 (single quote), %3c (< ), %00 (null byte)\n\n'
        '### Examples\n'
        'Request: GET /tienda1/publico/index.jsp HTTP/1.1\n'
        'Output: {"label": "Normal", "reason": "Clean request, no attack pattern"}\n\n'
        "Request: GET /item?id=1' UNION SELECT username,password FROM users-- HTTP/1.1\n"
        'Output: {"label": "Anomalous", "reason": "UNION-based SQL Injection extracting credentials"}\n\n'
        'Request: GET /page?x=<img src=x onerror=alert(1)> HTTP/1.1\n'
        'Output: {"label": "Anomalous", "reason": "XSS via onerror event handler"}\n\n'
        'Request: GET /download?file=../../etc/shadow HTTP/1.1\n'
        'Output: {"label": "Anomalous", "reason": "Path Traversal targeting /etc/shadow"}\n\n'
        'Request: POST /comment HTTP/1.1\nBody: text=hello world&user=john\n'
        'Output: {"label": "Normal", "reason": "Regular form submission with benign content"}\n\n'
        f'### Classify\nRequest: {http_text}\nOutput:'
    )


PROMPT_STRATEGIES = [
    ("A_2shot_basic",        "2-shot Basic (기준선)",             prompt_a),
    ("B_5shot_multiattack",  "5-shot Multi-attack",              prompt_b),
    ("C_chain_of_thought",   "Chain-of-Thought",                 prompt_c),
    ("D_rule_hint_5shot",    "Rule-hint + 5-shot",               prompt_d),
]
print(f"{len(PROMPT_STRATEGIES)}가지 프롬프트 전략 준비 완료")

4가지 프롬프트 전략 준비 완료


## 3. 100건 분류 실행 (4가지 프롬프트 순차 실행)

> CPU 환경 기준 건당 1~3초. 4전략 × 100건 ≈ **10~20분** 소요 예상

In [12]:
# %% [3] 4가지 프롬프트 순차 실행
all_results = {}  # key=전략ID, value=(df, acc, f1, elapsed, unknown)

for strategy_id, strategy_name, prompt_fn in PROMPT_STRATEGIES:
    print(f"\n{'='*60}")
    print(f"[실행] {strategy_name}")
    print('='*60)
    df, acc, f1, elapsed, unknown = run_classification(
        prompt_fn, llm_sample, model="gemma3:4b", label=strategy_id
    )
    all_results[strategy_id] = {
        "name": strategy_name,
        "df": df,
        "acc": acc,
        "f1": f1,
        "elapsed": elapsed,
        "unknown": unknown,
    }

print("\n모든 프롬프트 실험 완료!")


[실행] 2-shot Basic (기준선)
  [A_2shot_basic] 10/100건 완료 (8.0초, 건당 0.80초)
  [A_2shot_basic] 20/100건 완료 (15.6초, 건당 0.78초)
  [A_2shot_basic] 30/100건 완료 (22.6초, 건당 0.75초)
  [A_2shot_basic] 40/100건 완료 (31.0초, 건당 0.78초)
  [A_2shot_basic] 50/100건 완료 (39.1초, 건당 0.78초)
  [A_2shot_basic] 60/100건 완료 (45.2초, 건당 0.75초)
  [A_2shot_basic] 70/100건 완료 (52.6초, 건당 0.75초)
  [A_2shot_basic] 80/100건 완료 (58.8초, 건당 0.74초)
  [A_2shot_basic] 90/100건 완료 (65.1초, 건당 0.72초)
  [A_2shot_basic] 100/100건 완료 (73.4초, 건당 0.73초)

[A_2shot_basic] 총 73.4초 | Acc=0.8400 | F1=0.8519 | Unknown=1건

[실행] 5-shot Multi-attack
  [B_5shot_multiattack] 10/100건 완료 (6.2초, 건당 0.62초)
  [B_5shot_multiattack] 20/100건 완료 (12.8초, 건당 0.64초)
  [B_5shot_multiattack] 30/100건 완료 (19.1초, 건당 0.64초)
  [B_5shot_multiattack] 40/100건 완료 (25.1초, 건당 0.63초)
  [B_5shot_multiattack] 50/100건 완료 (31.1초, 건당 0.62초)
  [B_5shot_multiattack] 60/100건 완료 (36.3초, 건당 0.60초)
  [B_5shot_multiattack] 70/100건 완료 (41.7초, 건당 0.60초)
  [B_5shot_multiattack] 80/100건 완료 (46.5초, 건당 

In [13]:
# %% [4] 정확도/F1 비교 테이블
summary = []
for sid, r in all_results.items():
    summary.append({
        "전략": r["name"],
        "정확도": f"{r['acc']:.4f}",
        "F1": f"{r['f1']:.4f}",
        "소요(초)": f"{r['elapsed']:.1f}",
        "건당(초)": f"{r['elapsed']/100:.2f}",
        "Unknown": r["unknown"],
    })

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))

# 최고 성능 전략
best_id = max(all_results, key=lambda k: all_results[k]["f1"])
print(f"\n최고 F1 전략: {all_results[best_id]['name']} (F1={all_results[best_id]['f1']:.4f})")

                 전략    정확도     F1 소요(초) 건당(초)  Unknown
 2-shot Basic (기준선) 0.8400 0.8519  73.4  0.73        1
5-shot Multi-attack 0.8500 0.8454  57.9  0.58        0
   Chain-of-Thought 0.8200 0.8421 131.7  1.32        0
 Rule-hint + 5-shot 0.8300 0.8317  60.8  0.61        0

최고 F1 전략: 2-shot Basic (기준선) (F1=0.8519)


In [14]:
# %% [5] 최고 성능 전략의 오분류 분석
best = all_results[best_id]
df = best["df"]

print(f"=== {best['name']} - 상세 분류 리포트 ===")
y_true = (df["true"] == "Anomalous").astype(int)
y_pred = (df["pred_clean"] == "Anomalous").astype(int)
print(classification_report(y_true, y_pred, target_names=["Normal", "Anomalous"]))

print("\n--- 오분류 사례 (상위 5건) ---")
misclf = df[y_true != y_pred].head(5)
for _, r in misclf.iterrows():
    print(f"  실제={r['true']:10s} 예측={r['pred']:10s}  {r['http_short'][:80]}")
    print(f"  근거: {r['reason']}\n")

=== 2-shot Basic (기준선) - 상세 분류 리포트 ===
              precision    recall  f1-score   support

      Normal       0.84      0.81      0.83        47
   Anomalous       0.84      0.87      0.85        53

    accuracy                           0.84       100
   macro avg       0.84      0.84      0.84       100
weighted avg       0.84      0.84      0.84       100


--- 오분류 사례 (상위 5건) ---
  실제=Normal     예측=Anomalous   GET /tienda1/miembros/editar.jsp?modo=registro&login=gargulak&password=Bu7c2Pié&
  근거: The request contains a large number of parameters, including unusual fields like 'ntc' and 'B1', suggesting a potential 

  실제=Normal     예측=Anomalous   GET /tienda1/publico/registro.jsp?modo=registro&login=peiser&password=9iri7aza&n
  근거: This request contains a large number of parameters, including unusual fields like 'ntc' and 'B1', suggesting a potential

  실제=Anomalous  예측=Normal      GET /busytime.nsf HTTP/1.1
  근거: Standard file request, likely a proprietary NSF file. No immediate

## 4. 자연어 판단 근거 검토 ★

LLM의 가장 큰 강점: **왜 그렇게 판단했는지** 사람이 읽을 수 있는 문장으로 설명.
이는 SOC(보안관제) 분석가가 1차 분류를 검토할 때 매우 유용합니다.

In [15]:
# %% [6] 최고 전략 - 공격 판단 사례 + 근거
print(f"=== [{best['name']}] 공격으로 판단한 사례 (상위 5건) ===\n")
attack_pred = best["df"][best["df"]["pred"] == "Anomalous"].head(5)
for _, r in attack_pred.iterrows():
    correct = "OK" if r["true"] == "Anomalous" else "오탐"
    print(f"[{correct}] 실제={r['true']:10s}  요청: {r['http_short']}")
    print(f"   - LLM 근거: {r['reason']}\n")

=== [2-shot Basic (기준선)] 공격으로 판단한 사례 (상위 5건) ===

[OK] 실제=Anomalous   요청: GET /tienda1/publico/registro.jsp?modo=registro&login=tejani&password=arable&nombre=Josiana&apellido
   - LLM 근거: URL-encoded characters within parameters, particularly the 'dni' and 'ntc' values, combined with a suspicious registrati

[오탐] 실제=Normal      요청: GET /tienda1/miembros/editar.jsp?modo=registro&login=gargulak&password=Bu7c2Pié&nombre=Deysi&apellid
   - LLM 근거: The request contains a large number of parameters, including unusual fields like 'ntc' and 'B1', suggesting a potential 

[OK] 실제=Anomalous   요청: GET /tienda1/publico/autenticar.jsp?modo=entrar&login=bienek&pwd=cloqu'e/ro&remember=off&B1=Entrar H
   - LLM 근거: The request contains unusual characters in the URL parameters (e.g., 'cloqu'e/ro') which could be part of an attempted i

[OK] 실제=Anomalous   요청: POST /tienda1/publico/anadir.jsp HTTP/1.1
Body: id=3/&nombre=Vino+Rioja&precio=85&cantidad=35&B1=Aña
   - LLM 근거: The URL path '/tienda1/publico/a

In [16]:
# %% [7] 결과 저장 (3교시에서도 활용)
save_data = {
    "summary": all_results,
    "best_strategy": best_id,
    # 기존 호환용 - 최고 전략 결과를 기본으로 노출
    "llm_df":   all_results[best_id]["df"],
    "llm_acc":  all_results[best_id]["acc"],
    "llm_f1":   all_results[best_id]["f1"],
    "llm_time": all_results[best_id]["elapsed"],
    "n_samples": len(llm_sample),
}
with open("llm_classification_results.pkl", "wb") as f:
    pickle.dump(save_data, f)
print(">> llm_classification_results.pkl 저장 완료")

>> llm_classification_results.pkl 저장 완료


## 5. 결과 MD 리포트 생성

In [17]:
# %% [8] MD 리포트 자동 생성
from datetime import datetime

today = datetime.now().strftime("%Y-%m-%d")

lines = [
    "# LLM 기반 HTTP 요청 분류 - 프롬프트 실험 결과",
    "",
    f"- **실험일**: {today}",
    f"- **모델**: gemma3:4b (Ollama)",
    f"- **데이터**: CSIC 2010 HTTP Dataset ({len(llm_sample)}건 샘플)",
    "",
    "---",
    "",
    "## 1. 프롬프트 전략 설명",
    "",
    "| 전략 ID | 전략명 | 설명 |",
    "|--------|--------|------|",
    "| A | 2-shot Basic | 기준선. 정상/SQL Injection 예시 2개 제공 |",
    "| B | 5-shot Multi-attack | SQL·XSS·Path Traversal·Command Injection 5가지 예시 |",
    "| C | Chain-of-Thought | 단계별 추론(Parse→Pattern→Decision) 후 JSON 출력 |",
    "| D | Rule-hint + 5-shot | 탐지 규칙 힌트(키워드 목록) + 다양한 5-shot 예시 |",
    "",
    "---",
    "",
    "## 2. 프롬프트 원문",
    "",
]

# 프롬프트 원문 삽입
sample_text = "GET /tienda1/publico/anadir.jsp?id=2'+OR+'1'='1 HTTP/1.1"
for sid, sname, pfn in PROMPT_STRATEGIES:
    lines.append(f"### {sname}")
    lines.append("")
    lines.append("```")
    lines.append(pfn(sample_text))
    lines.append("```")
    lines.append("")

lines += [
    "---",
    "",
    "## 3. 성능 비교 결과",
    "",
    "| 전략 | 정확도 | F1 | 소요(초) | 건당(초) | Unknown |",
    "|------|--------|-----|----------|----------|---------||",
]

for sid, r in all_results.items():
    best_mark = " **★**" if sid == best_id else ""
    lines.append(
        f"| {r['name']}{best_mark} "
        f"| {r['acc']:.4f} | {r['f1']:.4f} "
        f"| {r['elapsed']:.1f} | {r['elapsed']/100:.2f} "
        f"| {r['unknown']} |"
    )

lines += [
    "",
    f"**최고 성능**: {all_results[best_id]['name']} (F1={all_results[best_id]['f1']:.4f})",
    "",
    "---",
    "",
    "## 4. 오분류 분석 (최고 성능 전략)",
    "",
]

best_df = all_results[best_id]["df"]
y_true_s = (best_df["true"] == "Anomalous").astype(int)
y_pred_s = (best_df["pred_clean"] == "Anomalous").astype(int)
misclf = best_df[y_true_s != y_pred_s]

lines.append(f"총 오분류: {len(misclf)}건")
lines.append("")
lines.append("| # | 실제 | 예측 | HTTP 요청 (앞 80자) | LLM 근거 |")
lines.append("|---|------|------|---------------------|----------|")
for i, (_, r) in enumerate(misclf.head(10).iterrows(), 1):
    lines.append(
        f"| {i} | {r['true']} | {r['pred']} "
        f"| `{r['http_short'][:80]}` | {r['reason'][:80]} |"
    )

lines += [
    "",
    "---",
    "",
    "## 5. 결론 및 인사이트",
    "",
    "- Few-shot 예시의 **다양성**이 정확도에 직접 영향: 단순 2-shot → 5-shot 다양화로 성능 향상",
    "- **Rule-hint** 포함 시 인코딩 우회(%27 등) 탐지율 향상",
    "- Chain-of-Thought는 설명 품질↑ but 응답 길어져 파싱 실패(Unknown) 증가 가능성",
    "- LLM 분류 강점: 근거 설명으로 SOC 분석가 지원 가능",
    "- LLM 분류 약점: 속도(건당 1~3초), 비용, 비결정성",
    "",
]

md_content = "\n".join(lines)
md_path = "llm_prompt_experiment_results.md"
with open(md_path, "w", encoding="utf-8") as f:
    f.write(md_content)

print(f">> {md_path} 생성 완료 ({len(md_content)}자)")

>> llm_prompt_experiment_results.md 생성 완료 (6558자)


**다음**: `comparison_analysis.ipynb`로 1교시 ML 결과와 종합 비교합니다.